In [2]:
import os
import re
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd
from collections import defaultdict

from sympy import (
    symbols, Poly, fraction, simplify, Pow, log,
    Add, Abs, sin, cos, tan, sqrt, Dummy, Rational, exp,
    asin, acos, atan
)
from sympy.parsing.sympy_parser import (
    parse_expr,
    standard_transformations,
    implicit_multiplication_application,
    convert_xor
)

# Folder path and output file
folder_path = "/Users/guillermobautista/desktop/DHS1-G10/"
output_file = os.path.join(folder_path, "deparobefore.csv")

x, y = symbols("x y")
transformations = standard_transformations + (
    implicit_multiplication_application,
    convert_xor
)

# --- Helper functions ---

def extract_parenthesized(text, open_idx):
    depth = 0
    for i in range(open_idx, len(text)):
        if text[i] == '(':
            depth += 1
        elif text[i] == ')':
            depth -= 1
            if depth == 0:
                return text[open_idx + 1:i], i
    raise ValueError("Unmatched parenthesis")


def extract_bracketed(text, open_idx, open_char='[', close_char=']'):
    depth = 0
    for i in range(open_idx, len(text)):
        if text[i] == open_char:
            depth += 1
        elif text[i] == close_char:
            depth -= 1
            if depth == 0:
                return text[open_idx + 1:i], i
    raise ValueError("Unmatched bracket")


def split_top_level_args(s):
    parts = []
    depth = 0
    current = []

    for ch in s:
        if ch == '(':
            depth += 1
            current.append(ch)
        elif ch == ')':
            depth -= 1
            current.append(ch)
        elif ch == ',' and depth == 0:
            parts.append(''.join(current).strip())
            current = []
        else:
            current.append(ch)

    parts.append(''.join(current).strip())
    return parts


def split_top_level_commas_general(s):
    parts = []
    paren_depth = 0
    bracket_depth = 0
    current = []

    for ch in s:
        if ch == '(':
            paren_depth += 1
            current.append(ch)
        elif ch == ')':
            paren_depth -= 1
            current.append(ch)
        elif ch == '[':
            bracket_depth += 1
            current.append(ch)
        elif ch == ']':
            bracket_depth -= 1
            current.append(ch)
        elif ch == ',' and paren_depth == 0 and bracket_depth == 0:
            parts.append(''.join(current).strip())
            current = []
        else:
            current.append(ch)

    parts.append(''.join(current).strip())
    return parts


def normalize_inequality_symbols(s):
    return (
        s.replace("≤", "<=")
         .replace("≥", ">=")
         .replace("≦", "<=")
         .replace("≧", ">=")
    )


def is_domain_restriction(part):
    s = normalize_inequality_symbols(part.strip())

    if s.startswith("(") and s.endswith(")"):
        s = s[1:-1].strip()

    if "x" not in s:
        return False
    if not any(op in s for op in ["<", ">", "<=", ">="]):
        return False

    forbidden_formula_tokens = ["sin", "cos", "tan", "log", "ln", "lg", "sqrt", "^", "*", "/", "Abs", "abs"]
    if any(tok in s for tok in forbidden_formula_tokens):
        return False

    chained_patterns = [
        r'^.+?(<=|<)\s*x\s*(<=|<)\s*.+$',
        r'^.+?(>=|>)\s*x\s*(>=|>)\s*.+$',
        r'^x\s*(<=|<)\s*.+$',
        r'^x\s*(>=|>)\s*.+$',
        r'^.+\s*(<=|<)\s*x$',
        r'^.+\s*(>=|>)\s*x$'
    ]

    return any(re.fullmatch(p, s) for p in chained_patterns)


def strip_trailing_domain_restriction(expr):
    parts = split_top_level_commas_general(expr)
    if len(parts) >= 2:
        last_part = parts[-1].strip()
        if is_domain_restriction(last_part):
            return ','.join(parts[:-1]).strip()
    return expr.strip()


def strip_boolean_interval_factor(expr):
    s = expr.strip()

    changed = True
    while changed and s.startswith("(") and s.endswith(")"):
        changed = False
        depth = 0
        wraps_all = True
        for i, ch in enumerate(s):
            if ch == "(":
                depth += 1
            elif ch == ")":
                depth -= 1
                if depth == 0 and i != len(s) - 1:
                    wraps_all = False
                    break
        if wraps_all:
            s = s[1:-1].strip()
            changed = True

    factors = []
    depth = 0
    current = []
    for ch in s:
        if ch == "(":
            depth += 1
            current.append(ch)
        elif ch == ")":
            depth -= 1
            current.append(ch)
        elif ch == "*" and depth == 0:
            factors.append(''.join(current).strip())
            current = []
        else:
            current.append(ch)
    factors.append(''.join(current).strip())

    if len(factors) != 2:
        return expr.strip()

    f1, f2 = factors[0], factors[1]

    if is_domain_restriction(f1):
        return f2
    if is_domain_restriction(f2):
        return f1

    return expr.strip()


def starts_with_if_expression(text):
    if not text:
        return False
    s = text.strip()
    return bool(re.match(r'^\s*If\s*[\(\[]', s, flags=re.IGNORECASE))


def is_true_piecewise_if(text):
    """
    True only for genuine piecewise GeoGebra If-expressions, e.g.
      If(condition, expr1, expr2)
      If(condition1, expr1, condition2, expr2)
    but NOT for single-branch/domain-restricted forms like
      If(condition, expr)
    """
    if not starts_with_if_expression(text):
        return False

    s = text.strip()

    try:
        if s.startswith("If["):
            inside, _ = extract_bracketed(s, s.find("["), "[", "]")
        elif s.startswith("If("):
            inside, _ = extract_parenthesized(s, s.find("("))
        else:
            return False

        parts = split_top_level_commas_general(inside)
        return len(parts) >= 3

    except Exception:
        return False


def unwrap_if_rhs(rhs):
    rhs = rhs.strip()

    if rhs.startswith("If["):
        inside, _ = extract_bracketed(rhs, rhs.find("["), "[", "]")
        parts = split_top_level_commas_general(inside)
        if len(parts) >= 2:
            return parts[1].strip()

    if rhs.startswith("If("):
        inside, _ = extract_parenthesized(rhs, rhs.find("("))
        parts = split_top_level_commas_general(inside)
        if len(parts) >= 2:
            return parts[1].strip()

    return rhs


def clean_expression(expr):
    expr = expr.strip()

    if ":" in expr:
        left, right = expr.split(":", 1)
        if "=" in right:
            expr = right.strip()

    match = re.match(r'\s*[^=\s]+\(x\)\s*=\s*(.+)$', expr)
    if match:
        rhs = match.group(1).strip()

        if is_true_piecewise_if(rhs):
            return rhs

        rhs = unwrap_if_rhs(rhs)
        rhs = strip_trailing_domain_restriction(rhs)
        rhs = strip_boolean_interval_factor(rhs)
        return rhs

    match = re.match(r'\s*y\s*=\s*(.+)$', expr, flags=re.IGNORECASE)
    if match:
        rhs = match.group(1).strip()

        if is_true_piecewise_if(rhs):
            return rhs

        rhs = unwrap_if_rhs(rhs)
        rhs = strip_trailing_domain_restriction(rhs)
        rhs = strip_boolean_interval_factor(rhs)
        return rhs

    if "=" in expr:
        parts = expr.split("=", 1)
        rhs = parts[1].strip()

        if is_true_piecewise_if(rhs):
            return rhs

        rhs = unwrap_if_rhs(rhs)
        rhs = strip_trailing_domain_restriction(rhs)
        rhs = strip_boolean_interval_factor(rhs)
        return rhs

    if is_true_piecewise_if(expr):
        return expr

    expr = unwrap_if_rhs(expr)
    expr = strip_trailing_domain_restriction(expr)
    expr = strip_boolean_interval_factor(expr)
    return expr


def starts_with_explicit_y_equals(expr_text):
    return bool(re.match(r'^\s*y\s*=', expr_text, flags=re.IGNORECASE))


def is_explicit_y_function(expr_text):
    """
    Returns True only if the expression is genuinely of the form y = f(x),
    meaning it starts with y = and the right-hand side contains no y.
    """
    if not expr_text:
        return False

    if not starts_with_explicit_y_equals(expr_text):
        return False

    rhs = clean_expression(expr_text)
    return not re.search(r'\by\b', rhs)


def normalize_inverse_trig_notation(expr):
    """
    Converts common GeoGebra inverse trig notation to SymPy notation:
      sin^{-1}(x), sin^(-1)(x), sin^-1(x), sin⁻¹(x), arcsin(x) -> asin(x)
      cos^{-1}(x), cos^(-1)(x), cos^-1(x), cos⁻¹(x), arccos(x) -> acos(x)
      tan^{-1}(x), tan^(-1)(x), tan^-1(x), tan⁻¹(x), arctan(x) -> atan(x)
    """
    expr = expr.strip()

    patterns = [
        (r'\bsin\s*\^\s*\{\s*-\s*1\s*\}\s*\(', 'asin('),
        (r'\bcos\s*\^\s*\{\s*-\s*1\s*\}\s*\(', 'acos('),
        (r'\btan\s*\^\s*\{\s*-\s*1\s*\}\s*\(', 'atan('),

        (r'\bsin\s*\^\s*\(\s*-\s*1\s*\)\s*\(', 'asin('),
        (r'\bcos\s*\^\s*\(\s*-\s*1\s*\)\s*\(', 'acos('),
        (r'\btan\s*\^\s*\(\s*-\s*1\s*\)\s*\(', 'atan('),

        (r'\bsin\s*\^\s*-\s*1\s*\(', 'asin('),
        (r'\bcos\s*\^\s*-\s*1\s*\(', 'acos('),
        (r'\btan\s*\^\s*-\s*1\s*\(', 'atan('),

        (r'\bsin\s*⁻¹\s*\(', 'asin('),
        (r'\bcos\s*⁻¹\s*\(', 'acos('),
        (r'\btan\s*⁻¹\s*\(', 'atan('),

        (r'\barcsin\s*\(', 'asin('),
        (r'\barccos\s*\(', 'acos('),
        (r'\barctan\s*\(', 'atan('),
    ]

    for pattern, repl in patterns:
        expr = re.sub(pattern, repl, expr, flags=re.IGNORECASE)

    return expr


def replace_log_with_base_underscore(expr):
    pattern1 = re.compile(r'log_\{\s*([^{}]+?)\s*\}\s*\(')
    pattern2 = re.compile(r'log_([A-Za-z0-9\.\+\-\*/]+)\s*\(')
    pattern3 = re.compile(r'log_\(\s*([^\(\)]+?)\s*\)\s*\(')

    while True:
        m = pattern1.search(expr)
        if not m:
            break
        base = m.group(1).strip()
        start = m.start()
        open_paren = m.end() - 1
        arg, close_idx = extract_parenthesized(expr, open_paren)
        replacement = f'log({arg},{base})'
        expr = expr[:start] + replacement + expr[close_idx + 1:]

    while True:
        m = pattern3.search(expr)
        if not m:
            break
        base = m.group(1).strip()
        start = m.start()
        open_paren = m.end() - 1
        arg, close_idx = extract_parenthesized(expr, open_paren)
        replacement = f'log({arg},{base})'
        expr = expr[:start] + replacement + expr[close_idx + 1:]

    while True:
        m = pattern2.search(expr)
        if not m:
            break
        base = m.group(1).strip()
        start = m.start()
        open_paren = m.end() - 1
        arg, close_idx = extract_parenthesized(expr, open_paren)
        replacement = f'log({arg},{base})'
        expr = expr[:start] + replacement + expr[close_idx + 1:]

    return expr


def looks_like_numeric_constant(s):
    s = s.strip()
    if not s:
        return False
    if "x" in s.lower():
        return False
    return bool(re.fullmatch(r'[-+]?\d*\.?\d+([eE][-+]?\d+)?', s))


def replace_log_two_argument_form(expr):
    i = 0
    result = []

    while i < len(expr):
        if expr[i:i+4].lower() == 'log(':
            open_idx = i + 3
            inside, close_idx = extract_parenthesized(expr, open_idx)
            parts = split_top_level_args(inside)

            if len(parts) == 2:
                first, second = parts[0].strip(), parts[1].strip()

                if looks_like_numeric_constant(first) and "x" in second.lower():
                    result.append(f'log({second},{first})')
                else:
                    result.append(f'log({inside})')
            else:
                result.append(f'log({inside})')

            i = close_idx + 1
        else:
            result.append(expr[i])
            i += 1

    return ''.join(result)


def preprocess_for_sympy(expr):
    expr = expr.strip()

    expr = normalize_inverse_trig_notation(expr)

    expr = re.sub(r'\blg\s*\(', 'log(', expr, flags=re.IGNORECASE)
    expr = re.sub(r'\bln\s*\(', 'log(', expr, flags=re.IGNORECASE)
    expr = re.sub(r'\babs\s*\(', 'Abs(', expr, flags=re.IGNORECASE)

    expr = replace_log_with_base_underscore(expr)
    expr = replace_log_two_argument_form(expr)

    return expr


def replace_log_calls_with_placeholder(expr, placeholder="L"):
    result = []
    i = 0

    while i < len(expr):
        if re.match(r'(?<![A-Za-z0-9_])log\s*\(', expr[i:], flags=re.IGNORECASE):
            m = re.match(r'log\s*\(', expr[i:], flags=re.IGNORECASE)
            open_idx = i + m.end() - 1
            _, close_idx = extract_parenthesized(expr, open_idx)
            result.append(placeholder)
            i = close_idx + 1
        else:
            result.append(expr[i])
            i += 1

    return ''.join(result)


def replace_trig_calls_with_placeholder(expr, func_name, placeholder="T"):
    result = []
    i = 0
    pattern = re.compile(rf'(?<![A-Za-z0-9_]){re.escape(func_name)}\s*\(', flags=re.IGNORECASE)

    while i < len(expr):
        m = pattern.match(expr, i)
        if m:
            open_idx = m.end() - 1
            _, close_idx = extract_parenthesized(expr, open_idx)
            result.append(placeholder)
            i = close_idx + 1
        else:
            result.append(expr[i])
            i += 1

    return ''.join(result)


def contains_inverse_trig_str(expr):
    expr_pre = normalize_inverse_trig_notation(expr).replace(" ", "").lower()
    return any(tok in expr_pre for tok in ["asin(", "acos(", "atan("])


def is_pure_single_trig_expression_str(expr, trig_name):
    expr_pre = preprocess_for_sympy(expr)
    expr_pre = expr_pre.replace(" ", "")
    lower = expr_pre.lower()

    if any(tok in lower for tok in ["asin(", "acos(", "atan("]):
        return False

    if not re.search(rf'(?<![A-Za-z0-9_]){re.escape(trig_name)}\(', lower):
        return False

    other_trigs = {"sin", "cos", "tan"} - {trig_name}
    forbidden = [f"{name}(" for name in other_trigs] + ["log(", "sqrt(", "abs(", "exp(", "asin(", "acos(", "atan("]
    if any(tok in lower for tok in forbidden):
        return False

    replaced = replace_trig_calls_with_placeholder(expr_pre, trig_name, "T")

    stripped = re.sub(r'[-+]?\d*\.?\d+([eE][-+]?\d+)?', '', replaced)
    stripped = stripped.replace("T", "")
    stripped = re.sub(r'[\+\-\*/\^\(\),]', '', stripped)

    return stripped.strip() == ''


def is_pure_log_expression_str(expr):
    expr_pre = preprocess_for_sympy(expr)
    expr_pre = expr_pre.replace(" ", "")

    if "log(" not in expr_pre.lower():
        return False

    lower = expr_pre.lower()
    forbidden = ["sin(", "cos(", "tan(", "sqrt(", "abs(", "exp(", "asin(", "acos(", "atan("]
    if any(tok in lower for tok in forbidden):
        return False

    replaced = replace_log_calls_with_placeholder(expr_pre, "L")

    stripped = re.sub(r'[-+]?\d*\.?\d+([eE][-+]?\d+)?', '', replaced)
    stripped = stripped.replace("L", "")
    stripped = re.sub(r'[\+\-\*/\^\(\),]', '', stripped)

    return "x" not in stripped.lower()


def remove_constant_addition(expr_sym):
    var_terms = [t for t in Add.make_args(expr_sym) if t.has(x)]
    if not var_terms:
        return 0
    return simplify(Add(*var_terms))


def split_scalar_and_core(expr_sym):
    coeff, rest = expr_sym.as_independent(x, as_Add=False)
    return simplify(coeff), simplify(rest)


def substitute_x_in_expression(expr, arg_str):
    return re.sub(r'\bx\b', f'({arg_str})', expr)


def implicit_relation_to_y_expression(expr_text):
    """
    Converts an implicit line relation such as
    (3.05*x) - (3.06*y) = -1.81
    into an explicit expression for y.
    """
    expr_text = expr_text.strip()

    if re.match(r'^\s*y\s*=', expr_text, flags=re.IGNORECASE):
        if is_explicit_y_function(expr_text):
            return clean_expression(expr_text)
        return ""

    if "=" not in expr_text or "y" not in expr_text:
        return clean_expression(expr_text)

    try:
        left_str, right_str = expr_text.split("=", 1)
        left_pre = preprocess_for_sympy(left_str.strip())
        right_pre = preprocess_for_sympy(right_str.strip())

        local_dict = {
            "x": x, "y": y, "log": log, "Abs": Abs,
            "sin": sin, "cos": cos, "tan": tan, "sqrt": sqrt, "exp": exp,
            "asin": asin, "acos": acos, "atan": atan
        }

        left_expr = parse_expr(left_pre, transformations=transformations, local_dict=local_dict)
        right_expr = parse_expr(right_pre, transformations=transformations, local_dict=local_dict)

        eq_expr = simplify(left_expr - right_expr)
        poly_y = Poly(eq_expr, y)

        if poly_y.degree() == 1:
            a = poly_y.nth(1)
            b = poly_y.nth(0)
            if a != 0:
                y_expr = simplify(-b / a)
                return str(y_expr)

    except Exception:
        pass

    return ""


def get_referenced_object_expression(label, expressions_by_label, elements_by_label):
    """
    Returns an explicit expression in x for a referenced label if possible.
    Only returns expressions that are genuinely functions of x.
    """
    exp_info = expressions_by_label.get(label, {})
    elem_info = elements_by_label.get(label, {})

    expr_text = exp_info.get("exp", "")
    obj_type = elem_info.get("type", exp_info.get("type", ""))

    if expr_text:
        if obj_type == "line":
            return implicit_relation_to_y_expression(expr_text)

        elif obj_type == "conic":
            if is_explicit_y_function(expr_text):
                return clean_expression(expr_text)
            return ""

        else:
            return clean_expression(expr_text)

    if obj_type == "line":
        coords = elem_info.get("coords", None)
        if coords is not None:
            A, B, C = coords
            try:
                A = float(A)
                B = float(B)
                C = float(C)
                if abs(B) > 1e-12:
                    return str(simplify((-(A * x) - C) / B))
            except Exception:
                pass

    return ""


def resolve_function_references(expr, expressions_by_label, elements_by_label, max_depth=20):
    """
    Resolves referenced GeoGebra function calls such as:
      f(x)
      g(-x)
      p_{17}(x)
      eq1(x) where eq1 may be a line or conic

    Only substitutes referenced objects that can be expressed as genuine
    functions of x.
    """
    if not expressions_by_label and not elements_by_label:
        return expr

    builtins = {
        "sin", "cos", "tan", "log", "sqrt", "abs", "Abs", "ln", "lg", "exp", "If",
        "asin", "acos", "atan"
    }
    expr_current = expr.strip()

    for _ in range(max_depth):
        changed = False
        i = 0
        pieces = []

        while i < len(expr_current):
            m = re.match(r'([A-Za-z][A-Za-z0-9_{}]*)\s*\(', expr_current[i:])
            if not m:
                pieces.append(expr_current[i])
                i += 1
                continue

            func_name = m.group(1)
            absolute_start = i
            open_paren_idx = i + m.end() - 1

            if func_name in builtins:
                pieces.append(expr_current[i:open_paren_idx + 1])
                i = open_paren_idx + 1
                continue

            try:
                arg_inside, close_idx = extract_parenthesized(expr_current, open_paren_idx)
            except Exception:
                pieces.append(expr_current[i])
                i += 1
                continue

            full_call = expr_current[absolute_start:close_idx + 1]

            replacement_expr = get_referenced_object_expression(
                func_name, expressions_by_label, elements_by_label
            )

            if replacement_expr:
                replacement_expr = resolve_function_references(
                    replacement_expr, expressions_by_label, elements_by_label, max_depth=5
                )
                substituted = substitute_x_in_expression(replacement_expr, arg_inside)
                pieces.append(f"({substituted})")
                changed = True
            else:
                pieces.append(full_call)

            i = close_idx + 1

        new_expr = ''.join(pieces)
        if not changed or new_expr == expr_current:
            break
        expr_current = new_expr

    return expr_current


# --- Family detectors ---

def is_constant_function(expr_sym):
    return not expr_sym.has(x)


def is_affine_in_x(expr):
    try:
        expr = simplify(expr)
        poly = Poly(expr, x)
        return poly.degree() <= 1
    except Exception:
        return False


def is_exponential_core(core):
    rewritten = simplify(core.rewrite(Pow))

    if isinstance(rewritten, Pow):
        base, exponent = rewritten.args

        if base.has(x):
            return False
        if not exponent.has(x):
            return False
        if not is_affine_in_x(exponent):
            return False

        return True

    return False


def is_logarithmic_core(core):
    core = simplify(core)

    if core.has(sin) or core.has(cos) or core.has(tan) or core.has(Abs) or core.has(asin) or core.has(acos) or core.has(atan):
        return False

    if isinstance(core, Pow):
        base, exponent = core.args
        if not exponent.has(x) and isinstance(base, log) and base.args[0].has(x):
            return True

    log_atoms = [a for a in core.atoms(log) if len(a.args) >= 1 and a.args[0].has(x)]

    if not log_atoms:
        return False

    replacements = {}
    for i, atom in enumerate(log_atoms):
        replacements[atom] = Dummy(f"L{i}")
    replaced = core.xreplace(replacements)

    return not replaced.has(x)


def is_sine_core(core):
    core = simplify(core)

    if core.has(cos) or core.has(tan) or core.has(log) or core.has(Abs) or core.has(asin) or core.has(acos) or core.has(atan):
        return False

    sin_atoms = [a for a in core.atoms(sin) if a.args and a.args[0].has(x)]
    if not sin_atoms:
        return False

    dummies = []
    replacements = {}
    for i, atom in enumerate(sin_atoms):
        d = Dummy(f"S{i}")
        dummies.append(d)
        replacements[atom] = d

    replaced = core.xreplace(replacements)
    return replaced.free_symbols.issubset(set(dummies))


def is_cosine_core(core):
    core = simplify(core)

    if core.has(sin) or core.has(tan) or core.has(log) or core.has(Abs) or core.has(asin) or core.has(acos) or core.has(atan):
        return False

    cos_atoms = [a for a in core.atoms(cos) if a.args and a.args[0].has(x)]
    if not cos_atoms:
        return False

    dummies = []
    replacements = {}
    for i, atom in enumerate(cos_atoms):
        d = Dummy(f"C{i}")
        dummies.append(d)
        replacements[atom] = d

    replaced = core.xreplace(replacements)
    return replaced.free_symbols.issubset(set(dummies))


def is_tangent_core(core):
    core = simplify(core)

    if core.has(sin) or core.has(cos) or core.has(log) or core.has(Abs) or core.has(asin) or core.has(acos) or core.has(atan):
        return False

    tan_atoms = [a for a in core.atoms(tan) if a.args and a.args[0].has(x)]
    if not tan_atoms:
        return False

    dummies = []
    replacements = {}
    for i, atom in enumerate(tan_atoms):
        d = Dummy(f"T{i}")
        dummies.append(d)
        replacements[atom] = d

    replaced = core.xreplace(replacements)
    return replaced.free_symbols.issubset(set(dummies))


def is_sqrt_of_square(expr):
    if isinstance(expr, Pow):
        base, expn = expr.args
        if expn == Rational(1, 2) and isinstance(base, Pow):
            inner_base, inner_exp = base.args
            if inner_exp == 2 and inner_base.has(x):
                return True
    return False


def is_absolute_core(core):
    if core.func == Abs and core.args and core.args[0].has(x):
        return True
    if is_sqrt_of_square(core):
        return True
    return False


def is_radical_core(core):
    if isinstance(core, Pow):
        base, exponent = core.args
        if base.has(x) and exponent.is_Rational and exponent.q > 1:
            if is_sqrt_of_square(core):
                return False
            return True
    return False


def has_special_structure(expr_sym):
    if expr_sym.has(sin) or expr_sym.has(cos) or expr_sym.has(tan) or expr_sym.has(log) or expr_sym.has(Abs):
        return True

    if expr_sym.has(asin) or expr_sym.has(acos) or expr_sym.has(atan):
        return True

    for p in expr_sym.atoms(Pow):
        base, exponent = p.args

        if exponent.has(x) and not base.has(x):
            return True

        if base.has(x) and exponent.is_Rational and exponent.q > 1:
            return True

        if base.has(x) and exponent.has(x) and not exponent.is_Integer:
            return True

    return False


def classify_polynomial_or_rational(expr_sym_raw, expr_sym_simplified):
    try:
        raw_num, raw_den = expr_sym_raw.as_numer_denom()
        raw_den_poly = Poly(raw_den, x)

        if raw_den_poly.degree() > 0:
            return "RAT"

        simp_num, simp_den = fraction(expr_sym_simplified)
        simp_num_poly = Poly(simp_num, x)
        simp_den_poly = Poly(simp_den, x)

        if simp_den_poly.degree() > 0:
            return "RAT"

        deg = simp_num_poly.degree()

        if deg == 0:
            return "CON"
        elif deg == 1:
            return "LIN"
        elif deg == 2:
            return "QUA"
        elif deg >= 3:
            return "POL"

    except Exception:
        pass

    return None


def classify_function_core(expr):
    expr_str = expr.strip()

    if is_true_piecewise_if(expr_str):
        return "OTF"

    if contains_inverse_trig_str(expr_str):
        return "OTF"

    if is_pure_log_expression_str(expr_str):
        return "LOG"
    if is_pure_single_trig_expression_str(expr_str, "sin"):
        return "SIN"
    if is_pure_single_trig_expression_str(expr_str, "cos"):
        return "COS"
    if is_pure_single_trig_expression_str(expr_str, "tan"):
        return "TAN"

    try:
        expr_pre = preprocess_for_sympy(expr_str)

        expr_sym = parse_expr(
            expr_pre,
            transformations=transformations,
            local_dict={
                "x": x,
                "y": y,
                "log": log,
                "Abs": Abs,
                "sin": sin,
                "cos": cos,
                "tan": tan,
                "sqrt": sqrt,
                "exp": exp,
                "asin": asin,
                "acos": acos,
                "atan": atan
            }
        )

        expr_sym_raw = expr_sym
        expr_sym = simplify(expr_sym)

        if expr_sym.has(asin) or expr_sym.has(acos) or expr_sym.has(atan):
            return "OTF"

        if is_constant_function(expr_sym):
            return "CON"

        core_without_constants = remove_constant_addition(expr_sym)

        if core_without_constants != 0:
            coeff, core = split_scalar_and_core(core_without_constants)

            if not coeff.has(x):
                if is_exponential_core(core):
                    return "EXP"
                elif is_logarithmic_core(core):
                    return "LOG"
                elif is_sine_core(core):
                    return "SIN"
                elif is_cosine_core(core):
                    return "COS"
                elif is_tangent_core(core):
                    return "TAN"
                elif is_absolute_core(core):
                    return "ABS"
                elif is_radical_core(core):
                    return "RAD"

        algebraic_family = classify_polynomial_or_rational(expr_sym_raw, expr_sym)
        if algebraic_family is not None:
            return algebraic_family

        if has_special_structure(expr_sym):
            return "OTF"

        return "OTF"

    except Exception:
        expr_fallback = normalize_inverse_trig_notation(preprocess_for_sympy(expr_str)).lower().replace(" ", "")

        if is_true_piecewise_if(expr_fallback):
            return "OTF"

        if any(tok in expr_fallback for tok in ["asin(", "acos(", "atan("]):
            return "OTF"

        if is_pure_log_expression_str(expr_fallback):
            return "LOG"
        if is_pure_single_trig_expression_str(expr_fallback, "sin"):
            return "SIN"
        if is_pure_single_trig_expression_str(expr_fallback, "cos"):
            return "COS"
        if is_pure_single_trig_expression_str(expr_fallback, "tan"):
            return "TAN"

        if re.fullmatch(r'[-+]?\d+(\.\d+)?', expr_fallback):
            return "CON"

        if re.fullmatch(r'[-+]?\d*\.?\d*\*?sin\([^\)]+\)([-+].+)?', expr_fallback):
            return "SIN"
        if re.fullmatch(r'[-+]?\d*\.?\d*\*?cos\([^\)]+\)([-+].+)?', expr_fallback):
            return "COS"
        if re.fullmatch(r'[-+]?\d*\.?\d*\*?tan\([^\)]+\)([-+].+)?', expr_fallback):
            return "TAN"

        if re.fullmatch(r'[-+]?\d*\.?\d*\*?abs\([^\)]+\)([-+].+)?', expr_fallback):
            return "ABS"

        if re.fullmatch(
            r'[-+]?\d*\.?\d*\*?exp\(([-+]?\d*\.?\d*\*?x)?([-+]\d*\.?\d+)?\)([-+].+)?',
            expr_fallback
        ):
            return "EXP"

        if re.fullmatch(
            r'[-+]?\d*\.?\d*\*?[a-z0-9\.]+\^\(?[-+]?\d*\.?\d*\*?x([-+]\d*\.?\d+)?\)?([-+].+)?',
            expr_fallback
        ):
            return "EXP"

        if "sqrt(" in expr_fallback:
            return "RAD"

        if "/" in expr_fallback and "x" in expr_fallback:
            return "RAT"

        if re.search(r'x(\*\*|\^)\(?2\)?', expr_fallback):
            return "QUA"
        elif re.search(r'x(\*\*|\^)[3-9]', expr_fallback):
            return "POL"
        elif "x" in expr_fallback and "/" not in expr_fallback:
            return "LIN"

        return "OTF"


def classify_function(expression, expressions_by_label=None, elements_by_label=None):
    expr = clean_expression(expression)

    if is_true_piecewise_if(expr):
        return "OTF"

    if expressions_by_label is not None or elements_by_label is not None:
        expr = resolve_function_references(expr, expressions_by_label, elements_by_label)

    if is_true_piecewise_if(expr):
        return "OTF"

    return classify_function_core(expr)


def classify_conic(matrix):
    A, B, C, D, E, F = matrix
    if B == 0 and A == C:
        return "conic_circle"
    elif B**2 - 4*A*C < 0:
        return "conic_ellipse"
    elif B**2 - 4*A*C == 0:
        return "conic_parabola"
    elif B**2 - 4*A*C > 0:
        return "conic_hyperbola"
    else:
        return "conic_other"


def parse_geogebra_file(file_path):
    with zipfile.ZipFile(file_path, 'r') as z:
        with z.open('geogebra.xml') as xml_file:
            tree = ET.parse(xml_file)
    return tree


# --- Main processing ---
all_object_types = set()
data = []

for filename in os.listdir(folder_path):
    if filename.endswith(".ggb"):
        file_path = os.path.join(folder_path, filename)
        try:
            tree = parse_geogebra_file(file_path)
            root = tree.getroot()

            counts = defaultdict(int)

            expressions_by_label = {}
            for exp_elem in root.findall(".//expression"):
                label = exp_elem.attrib.get("label", "")
                exp_type = exp_elem.attrib.get("type", "")
                exp_text = exp_elem.attrib.get("exp", "")
                expressions_by_label[label] = {"type": exp_type, "exp": exp_text}

            elements_by_label = {}
            for elem in root.findall(".//element"):
                label = elem.attrib.get("label", "")
                obj_type = elem.attrib.get("type", "")
                elem_info = {"type": obj_type}

                coords_elem = elem.find("coords")
                if coords_elem is not None:
                    elem_info["coords"] = (
                        coords_elem.attrib.get("x", "0"),
                        coords_elem.attrib.get("y", "0"),
                        coords_elem.attrib.get("z", "0")
                    )

                matrix_elem = elem.find(".//matrix")
                if matrix_elem is not None:
                    elem_info["matrix"] = {
                        "A0": matrix_elem.attrib.get("A0", 0),
                        "A1": matrix_elem.attrib.get("A1", 0),
                        "A2": matrix_elem.attrib.get("A2", 0),
                        "A3": matrix_elem.attrib.get("A3", 0),
                        "A4": matrix_elem.attrib.get("A4", 0),
                        "A5": matrix_elem.attrib.get("A5", 0)
                    }

                elements_by_label[label] = elem_info

            total_functions = 0

            construction = root.find(".//construction")
            if construction is not None:
                elements = construction.findall("element")
                commands = construction.findall("command")

                num_elements = len(elements)
                element_labels = [el.attrib.get("label", "") for el in elements]

                command_outputs = set()
                for cmd in commands:
                    for out in cmd.findall("output"):
                        for k, v in out.attrib.items():
                            if k.startswith("a"):
                                command_outputs.add(v)

                free_labels = [lab for lab in element_labels if lab and lab not in command_outputs]
                num_steps = len(free_labels) + len(commands)
            else:
                num_elements = 0
                num_steps = 0

            counts["NS"] = num_steps
            counts["NE"] = num_elements
            all_object_types.add("NS")
            all_object_types.add("NE")

            for elem in root.findall(".//element"):
                obj_type = elem.attrib.get("type", "")
                label = elem.attrib.get("label", "")
                exp_info = expressions_by_label.get(label, {})
                expr_text = exp_info.get("exp", "")

                if obj_type == "function":
                    total_functions += 1
                    if expr_text:
                        ftype = classify_function(expr_text, expressions_by_label, elements_by_label)
                        counts[ftype] += 1
                        all_object_types.add(ftype)
                    else:
                        counts["OTF"] += 1
                        all_object_types.add("OTF")

                elif obj_type == "line":
                    if expr_text and is_explicit_y_function(expr_text):
                        total_functions += 1
                        ftype = classify_function(expr_text, expressions_by_label, elements_by_label)
                        counts[ftype] += 1
                        all_object_types.add(ftype)
                    else:
                        counts["line"] += 1
                        all_object_types.add("line")

                elif obj_type == "conic":
                    if expr_text and is_explicit_y_function(expr_text):
                        total_functions += 1
                        ftype = classify_function(expr_text, expressions_by_label, elements_by_label)
                        counts[ftype] += 1
                        all_object_types.add(ftype)
                    else:
                        matrix_elem = elem.find(".//matrix")
                        if matrix_elem is not None:
                            try:
                                A = float(matrix_elem.attrib.get("A0", 0))
                                B = float(matrix_elem.attrib.get("A1", 0))
                                C = float(matrix_elem.attrib.get("A2", 0))
                                D = float(matrix_elem.attrib.get("A3", 0))
                                E = float(matrix_elem.attrib.get("A4", 0))
                                F = float(matrix_elem.attrib.get("A5", 0))
                                conic_type = classify_conic([A, B, C, D, E, F])
                                counts[conic_type] += 1
                                all_object_types.add(conic_type)
                            except Exception:
                                counts["conic_unknown"] += 1
                                all_object_types.add("conic_unknown")
                        else:
                            counts["conic_unknown"] += 1
                            all_object_types.add("conic_unknown")

                elif obj_type == "parametric":
                    counts["PAR"] += 1
                    all_object_types.add("PAR")

                else:
                    counts[obj_type] += 1
                    all_object_types.add(obj_type)

            counts["TOT"] = total_functions
            all_object_types.add("TOT")

            row = {"filename": filename}
            row.update(counts)
            data.append(row)

        except Exception as e:
            print(f"Skipping {filename}: {e}")

df = pd.DataFrame(data)

for obj in all_object_types:
    if obj not in df.columns:
        df[obj] = 0

df = df.fillna(0)

function_order = [
    "CON", "LIN", "ABS", "QUA", "POL", "RAT", "RAD",
    "SIN", "COS", "TAN", "EXP", "LOG", "OTF",
    "TOT", "PAR", "NS", "NE"
]

remaining_types = sorted([obj for obj in all_object_types if obj not in function_order])
final_columns = ["filename"] + function_order + remaining_types

for col in final_columns:
    if col not in df.columns:
        df[col] = 0

df = df[final_columns].sort_values("filename").reset_index(drop=True)

print(df)
df.to_csv(output_file, index=False)
print(f"GeoGebra object tally saved to: {output_file}")

# --- HTML Dashboard Output ---

dashboard_file = os.path.splitext(output_file)[0] + "_dashboard.html"

function_cols = [
    "CON", "LIN", "ABS", "QUA", "POL", "RAT", "RAD",
    "SIN", "COS", "TAN", "EXP", "LOG", "OTF"
]

existing_function_cols = [col for col in function_cols if col in df.columns]

summary = {
    "Number of applets": int(len(df)),
    "Total functions": int(df["TOT"].sum()) if "TOT" in df.columns else 0,
    "Average functions per applet": round(df["TOT"].mean(), 2) if "TOT" in df.columns and len(df) > 0 else 0,
    "Average construction steps": round(df["NS"].mean(), 2) if "NS" in df.columns and len(df) > 0 else 0,
    "Average elements": round(df["NE"].mean(), 2) if "NE" in df.columns and len(df) > 0 else 0
}

function_totals = {
    col: int(df[col].sum())
    for col in existing_function_cols
}

max_function_count = max(function_totals.values()) if function_totals else 1

function_rows = ""
for func, count in function_totals.items():
    width = (count / max_function_count) * 100 if max_function_count > 0 else 0
    function_rows += f"""
    <tr>
        <td>{func}</td>
        <td>{count}</td>
        <td>
            <div class="bar-container">
                <div class="bar" style="width:{width:.2f}%"></div>
            </div>
        </td>
    </tr>
    """

summary_cards = ""
for key, value in summary.items():
    summary_cards += f"""
    <div class="card">
        <div class="card-title">{key}</div>
        <div class="card-value">{value}</div>
    </div>
    """

applet_data = []
for _, row in df.iterrows():
    applet_data.append({
        "filename": row.get("filename", ""),
        "TOT": int(row.get("TOT", 0)),
        "NS": int(row.get("NS", 0)),
        "NE": int(row.get("NE", 0)),
        "functions": {
            col: int(row.get(col, 0))
            for col in existing_function_cols
        }
    })

applet_js_data = str(applet_data).replace("'", '"')

table_html = df.to_html(index=False, classes="data-table", border=0)

html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>GeoGebra Extraction Dashboard</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 30px;
            background-color: #f7f7f7;
            color: #222;
        }}

        h1, h2 {{
            color: #1f2937;
        }}

        .subtitle {{
            color: #555;
            margin-bottom: 30px;
        }}

        .cards {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
            gap: 15px;
            margin-bottom: 35px;
        }}

        .card {{
            background: white;
            padding: 18px;
            border-radius: 10px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.08);
        }}

        .card-title {{
            font-size: 14px;
            color: #666;
            margin-bottom: 8px;
        }}

        .card-value {{
            font-size: 28px;
            font-weight: bold;
            color: #111827;
        }}

        table {{
            border-collapse: collapse;
            width: 100%;
            background: white;
            margin-bottom: 35px;
            font-size: 14px;
        }}

        th, td {{
            padding: 8px 10px;
            border-bottom: 1px solid #ddd;
            text-align: left;
        }}

        th {{
            background-color: #e5e7eb;
            font-weight: bold;
        }}

        .bar-container {{
            width: 100%;
            background-color: #e5e7eb;
            border-radius: 4px;
            height: 18px;
        }}

        .bar {{
            height: 18px;
            background-color: #2563eb;
            border-radius: 4px;
        }}

        .table-wrapper {{
            overflow-x: auto;
        }}

        .applet-chart-box {{
            background: white;
            padding: 20px;
            border-radius: 12px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.08);
            margin-bottom: 35px;
        }}

        .applet-title {{
            text-align: center;
            font-size: 18px;
            font-weight: bold;
            margin-bottom: 8px;
        }}

        .applet-stats {{
            text-align: center;
            color: #555;
            margin-bottom: 20px;
        }}

        .chart-row {{
            display: grid;
            grid-template-columns: 70px 60px 1fr;
            gap: 10px;
            align-items: center;
            margin-bottom: 8px;
        }}

        .chart-label {{
            font-weight: bold;
        }}

        .chart-value {{
            text-align: right;
        }}

        .chart-bar-container {{
            background-color: #e5e7eb;
            height: 22px;
            border-radius: 4px;
        }}

        .chart-bar {{
            height: 22px;
            background-color: #2563eb;
            border-radius: 4px;
        }}

        .carousel-controls {{
            text-align: center;
            margin-top: 18px;
        }}

        .carousel-button {{
            font-size: 22px;
            padding: 8px 16px;
            margin: 0 8px;
            cursor: pointer;
            border: none;
            border-radius: 6px;
            background-color: #2563eb;
            color: white;
        }}

        .carousel-button:hover {{
            background-color: #1d4ed8;
        }}

        .note {{
            font-size: 13px;
            color: #666;
            margin-top: 20px;
        }}
    </style>
</head>

<body>
    <h1>GeoGebra Extraction Dashboard</h1>
    <div class="subtitle">
        Summary of extracted mathematical objects and function classifications from the CSV file.
    </div>

    <h2>Dataset Summary</h2>
    <div class="cards">
        {summary_cards}
    </div>

    <h2>Individual Applet Function Profile</h2>
    <div class="applet-chart-box">
        <div class="applet-title" id="appletTitle"></div>
        <div class="applet-stats" id="appletStats"></div>
        <div id="appletChart"></div>

        <div class="carousel-controls">
            <button class="carousel-button" onclick="previousApplet()">&#8592;</button>
            <button class="carousel-button" onclick="nextApplet()">&#8594;</button>
        </div>
    </div>

    <h2>Overall Function Family Counts</h2>
    <table>
        <thead>
            <tr>
                <th>Function Type</th>
                <th>Total Count</th>
                <th>Visual Summary</th>
            </tr>
        </thead>
        <tbody>
            {function_rows}
        </tbody>
    </table>

    <h2>Full Extracted Data</h2>
    <div class="table-wrapper">
        {table_html}
    </div>

    <div class="note">
        Generated automatically from: {os.path.basename(output_file)}
    </div>

    <script>
        const applets = {applet_js_data};
        let currentIndex = 0;

        function showApplet(index) {{
            if (applets.length === 0) {{
                document.getElementById("appletTitle").textContent = "No applets found.";
                return;
            }}

            const applet = applets[index];
            const functions = applet.functions;

            document.getElementById("appletTitle").textContent =
                (index + 1) + " of " + applets.length + ": " + applet.filename;

            document.getElementById("appletStats").textContent =
                "Total functions: " + applet.TOT +
                " | Construction steps: " + applet.NS +
                " | Elements: " + applet.NE;

            let maxCount = 1;
            for (const key in functions) {{
                if (functions[key] > maxCount) {{
                    maxCount = functions[key];
                }}
            }}

            let chartHTML = "";

            for (const key in functions) {{
                const count = functions[key];
                const width = maxCount > 0 ? (count / maxCount) * 100 : 0;

                chartHTML += `
                    <div class="chart-row">
                        <div class="chart-label">${{key}}</div>
                        <div class="chart-value">${{count}}</div>
                        <div class="chart-bar-container">
                            <div class="chart-bar" style="width:${{width}}%"></div>
                        </div>
                    </div>
                `;
            }}

            document.getElementById("appletChart").innerHTML = chartHTML;
        }}

        function previousApplet() {{
            currentIndex = (currentIndex - 1 + applets.length) % applets.length;
            showApplet(currentIndex);
        }}

        function nextApplet() {{
            currentIndex = (currentIndex + 1) % applets.length;
            showApplet(currentIndex);
        }}

        showApplet(currentIndex);
    </script>
</body>
</html>
"""

with open(dashboard_file, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"HTML dashboard saved to: {dashboard_file}")

                 filename   CON   LIN  ABS   QUA  POL  RAT  RAD  SIN  COS  \
0   material-bmfc3svz.ggb   0.0   0.0  0.0   4.0  3.0    0  1.0  0.0  0.0   
1   material-dxyc5vqk.ggb   4.0   2.0  0.0   4.0  0.0    0  0.0  0.0  0.0   
2   material-gjx2tjv9.ggb   2.0   0.0  2.0  10.0  0.0    0  0.0  0.0  0.0   
3   material-gpadcqsk.ggb   3.0   0.0  2.0  10.0  0.0    0  0.0  0.0  0.0   
4   material-hvhyyf79.ggb   3.0   0.0  2.0  10.0  0.0    0  0.0  0.0  0.0   
5   material-jhvjsnk3.ggb   2.0   0.0  2.0  11.0  0.0    0  0.0  0.0  0.0   
6   material-jk3qg8h6.ggb   1.0  12.0  0.0   0.0  0.0    0  0.0  0.0  0.0   
7   material-nueuppvc.ggb   3.0   0.0  2.0  11.0  0.0    0  0.0  0.0  0.0   
8   material-nxjy92v2.ggb   0.0   0.0  0.0  11.0  0.0    0  0.0  0.0  0.0   
9   material-qsccryeb.ggb   4.0   4.0  0.0   3.0  0.0    0  0.0  0.0  0.0   
10  material-rjwuv43q.ggb  30.0   0.0  0.0   0.0  0.0    0  0.0  0.0  0.0   
11  material-uvyjwczv.ggb   0.0   0.0  0.0   0.0  0.0    0  7.0  1.0  7.0   